# LeetCode 365 — Water and Jug Problem bằng Python

Link đề: https://leetcode.com/problems/water-and-jug-problem/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Math, Greatest Common Divisor, Bézout's Identity

Skeleton LeetCode:

```python
class Solution:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc (tên tham số trên LeetCode hiện là `x`, `y`, `target`; phiên bản cũ dùng `jug1Capacity`, `jug2Capacity`, `targetCapacity`); mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có hai bình dung tích `x` và `y` lít, nước vô hạn, ba thao tác (đổ đầy một bình, đổ cạn một bình, rót từ bình này sang bình kia đến khi bình nhận đầy hoặc bình đổ cạn); hỏi **tổng lượng nước trong hai bình** có thể đạt đúng `target` không.
2. Nắm các công cụ Python: **tuple làm trạng thái** `(i, j)`, `set` đánh dấu đã thăm, `collections.deque` cho **BFS**, ngăn xếp (list) cho **DFS**, `min` để mô phỏng phép rót, `math.gcd` và **thuật toán Euclid**.
3. Đi từ cách **vét cạn đồ thị trạng thái** (BFS/DFS trên mọi cặp `(i, j)`) → **liệt kê tổ hợp `a·x + b·y`** → **công thức gọn bằng `gcd`**: `target <= x + y` và `target % gcd(x, y) == 0` → **Euclid mở rộng** để *dựng bằng chứng* `a·x + b·y = target`.
4. Hiểu mấu chốt: **Đồng nhất thức Bézout** — các giá trị có dạng `a·x + b·y` (với `a, b` nguyên) chính là **các bội của `gcd(x, y)`**; cộng thêm điều kiện hiển nhiên `target <= x + y` (không thể chứa nhiều hơn sức chứa).
5. Tránh các lỗi kinh điển: quên điều kiện `target <= x + y`, chỉ kiểm tra `target % x` hay `target % y`, BFS/DFS không có tập `visited` (lặp vô hạn), đệ quy quá sâu (`RecursionError`).

## 2. Đề bài & ví dụ

Có hai bình nước, dung tích lần lượt `x` lít và `y` lít, và **nguồn nước vô hạn**. Trả về `true` nếu **tổng lượng nước trong cả hai bình** có thể đạt đúng `target` lít bằng các thao tác:

- **Đổ đầy** một trong hai bình (đổ đầy tới miệng bình).
- **Đổ cạn** một trong hai bình (đổ hết nước ra ngoài).
- **Rót** nước từ bình này sang bình kia cho đến khi **bình nhận đầy** hoặc **bình đổ cạn**.

**Ví dụ 1:** `x = 3`, `y = 5`, `target = 4` ⇒ `true`. Các bước (viết trạng thái `(nước bình 3 lít, nước bình 5 lít)`): đổ đầy bình 5 `(0, 5)`; rót sang bình 3 `(3, 2)`; đổ cạn bình 3 `(0, 2)`; rót 2 lít sang bình 3 `(2, 0)`; đổ đầy bình 5 `(2, 5)`; rót sang bình 3 cho đầy `(3, 4)`; đổ cạn bình 3 `(0, 4)` — bình 5 lít chứa đúng `4` lít. (Ví dụ "Die Hard" nổi tiếng.)

**Ví dụ 2:** `x = 2`, `y = 6`, `target = 5` ⇒ `false`.

**Ví dụ 3:** `x = 1`, `y = 2`, `target = 3` ⇒ `true` (đổ đầy cả hai bình: `1 + 2 = 3`).

**Ràng buộc:** `1 <= x, y, target <= 10^3`.

In [1]:
import math
import random
from collections import deque

examples = [
    (3, 5, 4, True),
    (2, 6, 5, False),
    (1, 2, 3, True),
]
for x, y, z, expected in examples:
    print(f"x={x}, y={y}, target={z} -> kỳ vọng {expected}")

x=3, y=5, target=4 -> kỳ vọng True
x=2, y=6, target=5 -> kỳ vọng False
x=1, y=2, target=3 -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Trạng thái `(i, j)` và tuple

Tình huống của hai bình tại một thời điểm hoàn toàn được mô tả bởi **hai số**: `i` lít trong bình 1, `j` lít trong bình 2 (`0 <= i <= x`, `0 <= j <= y`). Ta gói chúng thành **tuple** `(i, j)`. Tuple **bất biến** nên dùng được làm phần tử của `set` hoặc khóa `dict`, rất tiện để ghi nhớ "trạng thái đã thăm".

In [2]:
state = (3, 2)
i, j = state                      # "mở gói" tuple thành hai biến
print(i, j)                       # 3 2
visited = set()
visited.add((0, 0))
visited.add((3, 2))
print((3, 2) in visited, (2, 3) in visited)   # True False  ((3,2) và (2,3) là hai trạng thái KHÁC nhau)
print(len(visited))               # 2

3 2
True False
2


### 3.2. Sáu thao tác = sáu cạnh đi ra từ mỗi trạng thái

Từ `(i, j)` với dung tích `x`, `y` ta có tối đa **6 trạng thái kế tiếp**:

| Thao tác | Trạng thái mới |
|---|---|
| đổ đầy bình 1 | `(x, j)` |
| đổ đầy bình 2 | `(i, y)` |
| đổ cạn bình 1 | `(0, j)` |
| đổ cạn bình 2 | `(i, 0)` |
| rót bình 1 → bình 2 | `a = min(i, y - j)` rồi `(i - a, j + a)` |
| rót bình 2 → bình 1 | `b = min(j, x - i)` rồi `(i + b, j - b)` |

Lượng rót là `min(nước có để rót, chỗ trống ở bình nhận)`: đúng quy tắc "đến khi bình nhận đầy hoặc bình đổ cạn".

In [3]:
def next_states(i, j, x, y):
    a = min(i, y - j)             # rót 1 -> 2: tối đa i, nhưng bình 2 chỉ còn chỗ y - j
    b = min(j, x - i)             # rót 2 -> 1
    return [
        (x, j), (i, y),           # đổ đầy bình 1, bình 2
        (0, j), (i, 0),           # đổ cạn bình 1, bình 2
        (i - a, j + a),           # rót 1 -> 2
        (i + b, j - b),           # rót 2 -> 1
    ]


print(next_states(0, 0, 3, 5))    # [(3, 0), (0, 5), (0, 0), (0, 0), (0, 0), (0, 0)]
print(next_states(0, 5, 3, 5))    # từ (0,5): rót 2 -> 1 cho (3, 2)
print(next_states(3, 2, 3, 5))    # từ (3,2): rót 1 -> 2 cho (0, 5)

[(3, 0), (0, 5), (0, 0), (0, 0), (0, 0), (0, 0)]
[(3, 5), (0, 5), (0, 5), (0, 0), (0, 5), (3, 2)]
[(3, 2), (3, 5), (0, 2), (3, 0), (0, 5), (3, 2)]


### 3.3. BFS với `deque` và tập `visited`

**BFS** (duyệt theo bề rộng) dùng một **hàng đợi**: lấy trạng thái ở **đầu**, thêm các trạng thái kế tiếp **chưa thăm** vào **cuối**. `deque.popleft()` lấy đầu trong `O(1)`. Tập `visited` đảm bảo mỗi trạng thái chỉ vào hàng đợi **một lần** — không có nó, vì đồ thị có chu trình (đổ đầy rồi đổ cạn quay về chỗ cũ) ta sẽ **lặp vô hạn**.

In [4]:
q = deque([(0, 0)])
seen = {(0, 0)}
order = []
while q and len(order) < 6:                     # chỉ lấy 6 trạng thái đầu để xem
    i, j = q.popleft()
    order.append((i, j))
    for ns in next_states(i, j, 3, 5):
        if ns not in seen:
            seen.add(ns)
            q.append(ns)
print(order)                                   # [(0, 0), (3, 0), (0, 5), (3, 5), (0, 3), (3, 2)]

[(0, 0), (3, 0), (0, 5), (3, 5), (0, 3), (3, 2)]


### 3.4. Ước chung lớn nhất (GCD) và thuật toán Euclid

`gcd(a, b)` là số nguyên dương **lớn nhất chia hết cả `a` lẫn `b`**. **Euclid:** `gcd(a, b) = gcd(b, a % b)`, dừng khi `b = 0` (lúc đó `gcd = a`). Vì sao: mọi ước chung của `a` và `b` cũng chia hết `a − k·b`; nên tập ước chung của `(a, b)` và của `(b, a % b)` **giống hệt nhau**. Python có sẵn `math.gcd`.

In [5]:
def my_gcd(a, b):
    while b:
        a, b = b, a % b           # (a, b) -> (b, a mod b)
    return a


print(my_gcd(12, 18), math.gcd(12, 18))     # 6 6
print(my_gcd(3, 5), my_gcd(2, 6), my_gcd(7, 7))   # 1 2 7
# từng bước gcd(30, 18): (30,18) -> (18,12) -> (12,6) -> (6,0)
a, b = 30, 18
while b:
    print((a, b), "->", (b, a % b))
    a, b = b, a % b
print("gcd =", a)

6 6
1 2 7
(30, 18) -> (18, 12)
(18, 12) -> (12, 6)
(12, 6) -> (6, 0)
gcd = 6


### 3.5. Đồng nhất thức Bézout

**Bézout:** với số nguyên `x, y` không đồng thời bằng `0`, phương trình `a·x + b·y = g` (với `g = gcd(x, y)`) **có nghiệm nguyên** `(a, b)` (có thể âm). Hơn nữa, **mọi** số có dạng `a·x + b·y` đều **chia hết cho `g`** (vì `g` chia hết cả `x` và `y`), và ngược lại **mọi bội của `g`** đều viết được như vậy (nhân hai vế với `k`). Tóm lại:

`{ a·x + b·y : a, b nguyên } = { các bội của gcd(x, y) }`.

Ví dụ `x = 3`, `y = 5`: `gcd = 1` và `3·2 + 5·(−1) = 1`. Nên mọi số nguyên đều viết được dạng `3a + 5b` (ví dụ `4 = 3·3 + 5·(−1)`).

In [6]:
x, y = 3, 5
print(3 * 2 + 5 * (-1))                               # 1  -> gcd(3, 5) = 1 viết được
print(3 * 3 + 5 * (-1))                               # 4  -> target 4 viết được
vals = sorted({a * x + b * y for a in range(-10, 11) for b in range(-10, 11)})
print([v for v in vals if -3 <= v <= 12])             # mọi số nguyên -3..12 xuất hiện
x, y = 2, 6
vals = sorted({a * x + b * y for a in range(-10, 11) for b in range(-10, 11)})
print([v for v in vals if -3 <= v <= 12])             # chỉ các số CHẴN (bội của gcd = 2)

1
4
[-3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[-2, 0, 2, 4, 6, 8, 10, 12]


### 3.6. Toán tử `%` với số âm trong Python

Ở Python, `a % b` (với `b > 0`) luôn cho kết quả **không âm** trong `0..b−1`, kể cả khi `a` âm: `(-5) % 3 == 1`. Nhờ đó kiểm tra "`(z − a·x)` có chia hết cho `y` không" bằng `(z - a*x) % y == 0` đúng cả khi `z − a·x` âm (khác C/Java).

In [7]:
print(7 % 3, (-5) % 3, (-6) % 3)            # 1 1 0
print((4 - 3 * 3) % 5 == 0)                # 4 - 9 = -5 chia hết cho 5: True

1 1 0
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ta rót thử với `x = 3`, `y = 5`, `target = 4` theo đúng các bước của ví dụ 1 và theo dõi **tổng** `i + j` sau mỗi bước.

| Bước | Thao tác | Trạng thái `(i, j)` | Tổng `i + j` |
|---|---|---|---|
| 0 | bắt đầu | `(0, 0)` | 0 |
| 1 | đổ đầy bình 5 | `(0, 5)` | 5 |
| 2 | rót 5 → 3 | `(3, 2)` | 5 |
| 3 | đổ cạn bình 3 | `(0, 2)` | 2 |
| 4 | rót 5 → 3 | `(2, 0)` | 2 |
| 5 | đổ đầy bình 5 | `(2, 5)` | 7 |
| 6 | rót 5 → 3 | `(3, 4)` | 7 |
| 7 | đổ cạn bình 3 | `(0, 4)` | 4 ✓ |

Quan sát hai điều: (1) **phép rót không đổi tổng**; chỉ đổ đầy/đổ cạn làm tổng đổi. (2) Các tổng xuất hiện `0, 5, 2, 7, 4` đều có dạng `3a + 5b` với `a, b` nguyên: `5 = 3·0 + 5·1`, `2 = 3·4 + 5·(−2)`, `7 = 3·4 + 5·(−1)`, `4 = 3·3 + 5·(−1)`. Tức là tổng luôn là **tổ hợp nguyên của `x` và `y`**. Bình ví dụ `x = 2`, `y = 6`: mọi tổng đều chẵn, nên `target = 5` (lẻ) bất khả.

In [8]:
def apply_op(state, op, x, y):
    i, j = state
    if op == "fill1":
        return (x, j)
    if op == "fill2":
        return (i, y)
    if op == "empty1":
        return (0, j)
    if op == "empty2":
        return (i, 0)
    if op == "pour12":
        a = min(i, y - j)
        return (i - a, j + a)
    if op == "pour21":
        b = min(j, x - i)
        return (i + b, j - b)
    raise ValueError(op)


# Đúng 7 bước của ví dụ 1 (x = 3, y = 5)
state = (0, 0)
for step, op in enumerate(["fill2", "pour21", "empty1", "pour21", "fill2", "pour21", "empty1"], 1):
    state = apply_op(state, op, 3, 5)
    print(f"bước {step}: {op:7s} -> {state}, tổng = {sum(state)}")

bước 1: fill2   -> (0, 5), tổng = 5
bước 2: pour21  -> (3, 2), tổng = 5
bước 3: empty1  -> (0, 2), tổng = 2
bước 4: pour21  -> (2, 0), tổng = 2
bước 5: fill2   -> (2, 5), tổng = 7
bước 6: pour21  -> (3, 4), tổng = 7
bước 7: empty1  -> (0, 4), tổng = 4


## 5. Cách 1 — Brute Force / baseline: BFS trên đồ thị trạng thái

**Ý tưởng.** Coi mỗi `(i, j)` là một **đỉnh**, mỗi thao tác là một **cạnh**. Xuất phát từ `(0, 0)`, duyệt BFS mọi trạng thái đạt được và kiểm tra có trạng thái nào có `i + j == target` không. Vì từ trạng thái có `i == target` hoặc `j == target` ta luôn đổ cạn bình kia để được tổng `target`, nên chỉ cần kiểm tra **tổng** `i + j == target` (đúng với đề phát biểu "tổng lượng nước trong hai bình").

In [9]:
def can_measure_bfs(x, y, z):
    start = (0, 0)
    seen = {start}
    q = deque([start])
    while q:
        i, j = q.popleft()
        if i + j == z:                           # tổng lượng nước đạt target
            return True
        for ns in next_states(i, j, x, y):
            if ns not in seen:
                seen.add(ns)
                q.append(ns)
    return False


for x, y, z, expected in examples:
    print(can_measure_bfs(x, y, z), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** Đếm thô, có tối đa `(x + 1)(y + 1)` trạng thái `(i, j)`, mỗi trạng thái có `6` cạnh ⇒ cận trên `O(x·y)`. Nhưng thực tế **mọi trạng thái đạt được đều có ít nhất một bình đầy hoặc cạn** (sau đổ đầy: bình đầy; sau đổ cạn: bình cạn; sau rót: bình nhận đầy hoặc bình đổ cạn), nên chỉ có tối đa khoảng `2(x + y + 2)` trạng thái đạt được ⇒ **Time `O(x + y)`**, **Space `O(x + y)`** (tập `visited` và hàng đợi).

**Nút thắt:** dù đã gọn, ta vẫn phải **khám phá cả đồ thị** để trả lời một câu hỏi có/không, và đoạn "vì sao" vẫn mơ hồ. Cách tính tay ở mục 4 gợi ý có **luật đơn giản** đằng sau: tổng luôn là tổ hợp nguyên của `x` và `y`.

## 6. Ý tưởng thuật toán / bước đột phá

**Điều kiện cần.**

1. `target <= x + y`: hai bình cùng chứa tối đa `x + y` lít.
2. `target` phải chia hết cho `g = gcd(x, y)`: mọi lượng nước trong bình đều là tổ hợp nguyên `a·x + b·y` của `x` và `y` (đổ đầy cộng `x` hoặc `y`, đổ cạn/rót chỉ chuyển nước hoặc bỏ đi những lượng cũng là tổ hợp như vậy), nên chia hết cho `g` (Bézout, mục 3.5).

**Điều kiện đủ.** Ngược lại, nếu `target <= x + y` và `g | target` thì làm được. Cách đại ý: dùng chiến lược chuẩn — lặp lại "đổ đầy bình `x`, rót sang bình `y`; khi bình `y` đầy thì đổ cạn nó" — tổng lượng nước trong hệ cứ thay đổi theo bước `±x`, `±y` và quét qua mọi bội của `g` trong `[0, x + y]`. Chứng minh chặt chẽ cần chút số học; ở notebook này ta **kiểm chứng bằng vét cạn**: so sánh công thức với BFS trên **mọi** cặp `x, y <= 40` và mọi `target <= x + y + 3` (mục 12.3).

Kết luận: **`canMeasureWater ⇔ target <= x + y và target % gcd(x, y) == 0`**.

In [10]:
# Kiểm chứng nhanh công thức với 3 ví dụ của đề
for x, y, z, expected in examples:
    g = math.gcd(x, y)
    print(f"x={x} y={y} z={z}: gcd={g}, z<=x+y: {z <= x + y}, z%gcd==0: {z % g == 0} -> {z <= x + y and z % g == 0} (kỳ vọng {expected})")

x=3 y=5 z=4: gcd=1, z<=x+y: True, z%gcd==0: True -> True (kỳ vọng True)
x=2 y=6 z=5: gcd=2, z<=x+y: True, z%gcd==0: False -> False (kỳ vọng False)
x=1 y=2 z=3: gcd=1, z<=x+y: True, z%gcd==0: True -> True (kỳ vọng True)


## 7. Các cách tiếp theo

### Cách 2 — DFS bằng ngăn xếp tường minh (cùng đồ thị trạng thái, duyệt kiểu khác)

**Giải thích.** Cùng đồ thị như Cách 1, nhưng thay hàng đợi bằng **ngăn xếp** (`list` với `append`/`pop`): luôn đi sâu vào trạng thái vừa thêm. Lời giải mẫu viết DFS bằng **đệ quy** — gọn nhưng với `x, y` lớn độ sâu đệ quy có thể vượt giới hạn của Python (xem lỗi 4 ở mục 13), nên ta dùng ngăn xếp tường minh. Mỗi trạng thái được đánh dấu `visited` **ngay khi đẩy vào ngăn xếp**.

In [11]:
def can_measure_dfs(x, y, z):
    start = (0, 0)
    seen = {start}
    stack = [start]
    while stack:
        i, j = stack.pop()                       # lấy phần tử MỚI NHẤT (đi sâu)
        if i + j == z:
            return True
        for ns in next_states(i, j, x, y):
            if ns not in seen:
                seen.add(ns)
                stack.append(ns)
    return False


for x, y, z, expected in examples:
    print(can_measure_dfs(x, y, z), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDFS:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        seen = {(0, 0)}
        stack = [(0, 0)]
        while stack:
            i, j = stack.pop()
            if i + j == target:
                return True
            a = min(i, y - j)
            b = min(j, x - i)
            for ns in ((x, j), (i, y), (0, j), (i, 0), (i - a, j + a), (i + b, j - b)):
                if ns not in seen:
                    seen.add(ns)
                    stack.append(ns)
        return False


print(SolutionDFS().canMeasureWater(3, 5, 4), SolutionDFS().canMeasureWater(2, 6, 5))   # True False

True False


**Độ phức tạp Cách 2.** **Time `O(x + y)`**, **Space `O(x + y)`** (ngăn xếp + `visited`) — giống BFS (cùng lập luận: chỉ có `~2(x + y + 2)` trạng thái đạt được); khác ở thứ tự thăm: DFS không đảm bảo đường đi ngắn nhất.

### Cách 3 — Liệt kê tổ hợp `a·x + b·y = target`

**Giải thích từng đoạn.**

1. Điều kiện cần hiển nhiên: `target <= x + y`; nếu sai trả `False` ngay.
2. Cần có số nguyên `a, b` (có thể âm) với `a·x + b·y = target`. Với mỗi `a`, `b` tồn tại nguyên khi `(target − a·x)` **chia hết cho `y`**.
3. Chỉ cần thử `a = 0, 1, ..., y − 1`: vì `a·x mod y` **lặp lại** sau tối đa `y` bước (đổi `a` thành `a + y` làm `a·x` đổi `x·y`, chia hết cho `y`), các `a` khác không cho khả năng mới.

In [13]:
def can_measure_enum(x, y, z):
    if z > x + y:
        return False
    for a in range(y):                          # a·x mod y lặp lại với chu kỳ <= y
        if (z - a * x) % y == 0:                # b = (z - a*x) / y nguyên (có thể âm)
            return True
    return False


for x, y, z, expected in examples:
    print(can_measure_enum(x, y, z), "(kỳ vọng", expected, ")")
print([a for a in range(5) if (4 - a * 3) % 5 == 0])     # x=3, y=5, z=4: a = 3 (b = -1)

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )
[3]


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionEnum:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        if target > x + y:
            return False
        return any((target - a * x) % y == 0 for a in range(y))


print(SolutionEnum().canMeasureWater(3, 5, 4), SolutionEnum().canMeasureWater(2, 6, 5), SolutionEnum().canMeasureWater(1, 2, 3))

True False True


**Độ phức tạp Cách 3.** **Time `O(y)`** (vòng `a` từ `0` đến `y − 1`), **Space `O(1)`**. Nhanh hơn BFS nhiều (`y <= 1000` ⇒ nhiều nhất `1000` phép thử) và không cần biết khái niệm GCD.

### Cách 4 — Công thức GCD (Euclid tự viết)

**Giải thích.** Gộp hai nhận xét ở mục 6: `target <= x + y` và `target % gcd(x, y) == 0`. `gcd` tính bằng vòng lặp Euclid (`while b: a, b = b, a % b`) — `O(log min(x, y))`. Đây là lời giải ngắn nhất và nhanh nhất.

In [15]:
def can_measure_gcd(x, y, z):
    a, b = x, y
    while b:                                     # Euclid: gcd(a, b) = gcd(b, a % b)
        a, b = b, a % b
    g = a                                        # g = gcd(x, y)
    return z <= x + y and z % g == 0


for x, y, z, expected in examples:
    print(can_measure_gcd(x, y, z), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionGCD:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        if target > x + y:
            return False
        return target % math.gcd(x, y) == 0


print(SolutionGCD().canMeasureWater(3, 5, 4), SolutionGCD().canMeasureWater(2, 6, 5), SolutionGCD().canMeasureWater(1, 2, 3))

True False True


**Độ phức tạp Cách 4.** **Time `O(log min(x, y))`**, **Space `O(1)`**.

### Cách 5 — Euclid mở rộng: dựng "bằng chứng" `a·x + b·y = target`

**Giải thích.** Thay vì chỉ trả lời có/không, ta tìm **hệ số cụ thể** `(a, b)` với `a·x + b·y = target`. **Euclid mở rộng** `ext_gcd(p, q)` trả `(g, s, t)` thỏa `p·s + q·t = g`. Đệ quy: nếu `q = 0` thì `(p, 1, 0)`; ngược lại gọi `ext_gcd(q, p % q) = (g, s', t')` ⇒ vì `p % q = p − (p // q)·q` nên `g = q·s' + (p − (p//q)·q)·t' = p·t' + q·(s' − (p//q)·t')`; vậy `(g, t', s' − (p // q)·t')`. Rồi nhân hệ số với `k = target / g`. Nếu `a, b` ngược dấu thì ý nghĩa: "đổ đầy `|a|` lần một bình, đổ cạn `|b|` lần bình kia".

In [17]:
def ext_gcd(p, q):
    """Trả (g, s, t) với p*s + q*t = g = gcd(p, q)."""
    if q == 0:
        return p, 1, 0
    g, s1, t1 = ext_gcd(q, p % q)
    return g, t1, s1 - (p // q) * t1


def can_measure_certificate(x, y, z):
    """Trả (True, a, b) với a*x + b*y == z, hoặc (False, None, None)."""
    if z > x + y:
        return False, None, None
    g, s, t = ext_gcd(x, y)
    if z % g != 0:
        return False, None, None
    k = z // g
    return True, s * k, t * k


for x, y, z, expected in examples:
    ok, a, b = can_measure_certificate(x, y, z)
    print((x, y, z), "->", ok, f"a={a}, b={b}", "| kiểm tra:", (a * x + b * y == z) if ok else "-")
    assert ok == expected

(3, 5, 4) -> True a=8, b=-4 | kiểm tra: True
(2, 6, 5) -> False a=None, b=None | kiểm tra: -
(1, 2, 3) -> True a=3, b=0 | kiểm tra: True


Hãy đọc kết quả in ở ô trên (ví dụ 1 cho một cặp `(a, b)` thỏa `a·3 + b·5 = 4`): bất kỳ `(a, b)` nào thỏa phương trình đều là một "bằng chứng" hợp lệ (có vô số nghiệm: cộng `y` vào `a` và trừ `x` khỏi `b` giữ nguyên tổng). Phần ý nghĩa vật lý của nghiệm (đổ bao nhiêu lần) là hệ quả của Bézout, còn BFS (mục 8) cho ta **chuỗi thao tác cụ thể**.

In [18]:
# Code nộp LeetCode cho Cách 5 (chỉ trả bool, dùng Euclid mở rộng)
class SolutionExtGCD:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        def ext_gcd(p, q):
            if q == 0:
                return p, 1, 0
            g, s1, t1 = ext_gcd(q, p % q)
            return g, t1, s1 - (p // q) * t1

        if target > x + y:
            return False
        g, s, t = ext_gcd(x, y)
        return target % g == 0 and (s * (target // g)) * x + (t * (target // g)) * y == target


print(SolutionExtGCD().canMeasureWater(3, 5, 4), SolutionExtGCD().canMeasureWater(2, 6, 5))

True False


**Độ phức tạp Cách 5.** **Time `O(log min(x, y))`**, **Space `O(log min(x, y))`** cho ngăn xếp đệ quy của `ext_gcd` (độ sâu `~log` nên rất nhỏ).

## 8. TRACE từng bước

Trace **BFS có truy vết đường đi** cho `x = 3`, `y = 5`, `target = 4`: in từng trạng thái lấy ra, các trạng thái mới thêm vào, rồi dựng lại chuỗi thao tác từ `(0, 0)` đến trạng thái có tổng `4`.

In [19]:
def trace_bfs(x, y, z):
    start = (0, 0)
    parent = {start: None}
    q = deque([start])
    step = 0
    while q:
        i, j = q.popleft()
        step += 1
        print(f"#{step}: lấy {(i, j)}, tổng = {i + j}", end="")
        if i + j == z:
            print("  <-- đạt target!")
            path = []
            cur = (i, j)
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        new = []
        for ns in next_states(i, j, x, y):
            if ns not in parent:
                parent[ns] = (i, j)
                q.append(ns)
                new.append(ns)
        print(f"; thêm {new}")
    return None


path = trace_bfs(3, 5, 4)
print("Đường đi ngắn nhất:", path)
assert path[0] == (0, 0) and sum(path[-1]) == 4

#1: lấy (0, 0), tổng = 0; thêm [(3, 0), (0, 5)]
#2: lấy (3, 0), tổng = 3; thêm [(3, 5), (0, 3)]
#3: lấy (0, 5), tổng = 5; thêm [(3, 2)]
#4: lấy (3, 5), tổng = 8; thêm []
#5: lấy (0, 3), tổng = 3; thêm [(3, 3)]
#6: lấy (3, 2), tổng = 5; thêm [(0, 2)]
#7: lấy (3, 3), tổng = 6; thêm [(1, 5)]
#8: lấy (0, 2), tổng = 2; thêm [(2, 0)]
#9: lấy (1, 5), tổng = 6; thêm [(1, 0)]
#10: lấy (2, 0), tổng = 2; thêm [(2, 5)]
#11: lấy (1, 0), tổng = 1; thêm [(0, 1)]
#12: lấy (2, 5), tổng = 7; thêm [(3, 4)]
#13: lấy (0, 1), tổng = 1; thêm [(3, 1)]
#14: lấy (3, 4), tổng = 7; thêm [(0, 4)]
#15: lấy (3, 1), tổng = 4  <-- đạt target!
Đường đi ngắn nhất: [(0, 0), (3, 0), (0, 3), (3, 3), (1, 5), (1, 0), (0, 1), (3, 1)]


Mỗi mũi tên trong đường đi trên đều là một trong sáu thao tác (ta có thể suy ra bằng so sánh trạng thái liên tiếp). Đường BFS tìm được là **ngắn nhất** theo số thao tác (BFS duyệt theo tầng); có thể khác với đường 7 bước trong đề nhưng vẫn hợp lệ.

Bây giờ trace **công thức GCD** và **Euclid mở rộng** cho `x = 3`, `y = 5`, `target = 4`, và với `x = 2`, `y = 6`, `target = 5` để thấy vì sao `false`.

In [20]:
def trace_gcd(x, y, z):
    print(f"x={x}, y={y}, target={z}")
    a, b = x, y
    while b:
        print(f"  Euclid: ({a}, {b}) -> ({b}, {a % b})")
        a, b = b, a % b
    g = a
    print(f"  gcd = {g}")
    print(f"  z <= x + y ? {z} <= {x + y} -> {z <= x + y}")
    print(f"  z % gcd == 0 ? {z} % {g} = {z % g} -> {z % g == 0}")
    return z <= x + y and z % g == 0


print(trace_gcd(3, 5, 4))
print(trace_gcd(2, 6, 5))
print(trace_gcd(4, 6, 11))

x=3, y=5, target=4
  Euclid: (3, 5) -> (5, 3)
  Euclid: (5, 3) -> (3, 2)
  Euclid: (3, 2) -> (2, 1)
  Euclid: (2, 1) -> (1, 0)
  gcd = 1
  z <= x + y ? 4 <= 8 -> True
  z % gcd == 0 ? 4 % 1 = 0 -> True
True
x=2, y=6, target=5
  Euclid: (2, 6) -> (6, 2)
  Euclid: (6, 2) -> (2, 0)
  gcd = 2
  z <= x + y ? 5 <= 8 -> True
  z % gcd == 0 ? 5 % 2 = 1 -> False
False
x=4, y=6, target=11
  Euclid: (4, 6) -> (6, 4)
  Euclid: (6, 4) -> (4, 2)
  Euclid: (4, 2) -> (2, 0)
  gcd = 2
  z <= x + y ? 11 <= 10 -> False
  z % gcd == 0 ? 11 % 2 = 1 -> False
False


Bảng trace công thức:

| `(x, y, target)` | Euclid | `gcd` | `target <= x + y` | `target % gcd` | Kết quả |
|---|---|---|---|---|---|
| `(3, 5, 4)` | `(3,5)→(5,3)→(3,2)→(2,1)→(1,0)` | `1` | `4 <= 8` ✓ | `0` ✓ | `true` |
| `(2, 6, 5)` | `(2,6)→(6,2)→(2,0)` | `2` | `5 <= 8` ✓ | `1` ✗ | `false` |
| `(4, 6, 11)` | `(4,6)→(6,4)→(4,2)→(2,0)` | `2` | `11 <= 10` ✗ | | `false` |

Trace `ext_gcd(3, 5)` để thấy hệ số được "kéo ngược" từ đáy đệ quy lên:

In [21]:
def trace_ext_gcd(p, q, depth=0):
    pad = "  " * depth
    if q == 0:
        print(f"{pad}ext_gcd({p}, {q}) = ({p}, 1, 0)")
        return p, 1, 0
    print(f"{pad}ext_gcd({p}, {q}) -> gọi ext_gcd({q}, {p % q})")
    g, s1, t1 = trace_ext_gcd(q, p % q, depth + 1)
    s, t = t1, s1 - (p // q) * t1
    print(f"{pad}quay lại ext_gcd({p}, {q}): s={s}, t={t}, kiểm tra {p}*{s} + {q}*{t} = {p * s + q * t}")
    return g, s, t


g, s, t = trace_ext_gcd(3, 5)
assert 3 * s + 5 * t == g == 1

ext_gcd(3, 5) -> gọi ext_gcd(5, 3)
  ext_gcd(5, 3) -> gọi ext_gcd(3, 2)
    ext_gcd(3, 2) -> gọi ext_gcd(2, 1)
      ext_gcd(2, 1) -> gọi ext_gcd(1, 0)
        ext_gcd(1, 0) = (1, 1, 0)
      quay lại ext_gcd(2, 1): s=0, t=1, kiểm tra 2*0 + 1*1 = 1
    quay lại ext_gcd(3, 2): s=1, t=-1, kiểm tra 3*1 + 2*-1 = 1
  quay lại ext_gcd(5, 3): s=-1, t=2, kiểm tra 5*-1 + 3*2 = 1
quay lại ext_gcd(3, 5): s=2, t=-1, kiểm tra 3*2 + 5*-1 = 1


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 4 (GCD)**: ngắn và nhanh nhất. Ta dùng `math.gcd` (LeetCode cho phép `import math`).

In [22]:
import math


class Solution:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        if target > x + y:                    # hai bình cộng lại không chứa nổi target
            return False
        return target % math.gcd(x, y) == 0   # target phải là bội của gcd(x, y)


sol = Solution()
for x, y, z, expected in examples:
    print((x, y, z), "->", sol.canMeasureWater(x, y, z))
    assert sol.canMeasureWater(x, y, z) == expected

(3, 5, 4) -> True
(2, 6, 5) -> False
(1, 2, 3) -> True


## 10. Vì sao thuật toán đúng

**Điều kiện cần (rõ ràng).**

- Mỗi bình chứa nhiều nhất dung tích của nó, nên tổng `<= x + y`; nếu `target > x + y` thì **không** thể.
- Mọi lượng nước xuất hiện trong bình là **tổ hợp nguyên** `a·x + b·y` (qui nạp theo số thao tác: ban đầu `0`; đổ đầy đặt bình về `x` hoặc `y`; đổ cạn đặt về `0`; rót chuyển nước giữa hai bình, **tổng không đổi** và lượng mới trong mỗi bình là hiệu/ tổng các lượng cũ với dung tích). Mà tổ hợp nguyên của `x, y` luôn chia hết cho `g = gcd(x, y)`. Nên `target` (là tổng hai lượng ấy) chia hết cho `g`.

**Điều kiện đủ.** Nếu `target <= x + y` và `g | target`:

- Theo Bézout, tồn tại `a, b` nguyên sao cho `a·x + b·y = target` (nhân nghiệm của `a·x + b·y = g` với `target / g`).
- Chiến lược chuẩn "đổ đầy một bình, rót sang bình kia, khi bình kia đầy thì đổ cạn nó" sinh ra lần lượt mọi bội của `g` trong khoảng `[0, x + y]` làm tổng lượng nước (đây là chuỗi `Euclid` thực thể hóa). Ta **kiểm chứng bằng vét cạn** trên mọi `x, y <= 40` (mục 12.3) để không chỉ tin vào lập luận lời.

**BFS/DFS.** Đúng theo định nghĩa: duyệt **mọi** trạng thái đạt được từ `(0, 0)` qua 6 thao tác (tập `visited` đảm bảo dừng sau hữu hạn bước vì có tối đa `(x+1)(y+1)` trạng thái).

**Cách 3 (liệt kê).** `a·x + b·y = target` có nghiệm nguyên khi và chỉ khi tồn tại `a ∈ [0, y)` với `y | (target − a·x)` — vì `a ↦ a·x mod y` có chu kỳ chia hết `y`. Kết hợp `target <= x + y` cho đúng điều kiện bài.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS trên trạng thái | `O(x + y)` (cận thô `O(x·y)`) | `O(x + y)` |
| 2. DFS (ngăn xếp) | `O(x + y)` (cận thô `O(x·y)`) | `O(x + y)` |
| 3. Liệt kê tổ hợp `a·x + b·y` | `O(y)` | `O(1)` |
| 4. Công thức GCD | `O(log min(x, y))` | `O(1)` |
| 5. Euclid mở rộng | `O(log min(x, y))` | `O(log min(x, y))` |

Với `x, y <= 1000` mọi cách đều chạy được; Cách 1–2 tốn `~4000` trạng thái trong trường hợp xấu, Cách 3–5 gần như tức thì. Cách 1–2 còn giải được bài toán **mở rộng** (đường đi, số bước) mà công thức không trả lời.

## 12. Bộ test

Ta gom các cách vào `impls` (kể cả bản nộp). BFS là **oracle độc lập** (làm đúng theo định nghĩa thao tác).

In [23]:
sol = Solution()
impls = {
    "bfs": can_measure_bfs,
    "dfs": can_measure_dfs,
    "enum": can_measure_enum,
    "gcd": can_measure_gcd,
    "cert": lambda x, y, z: can_measure_certificate(x, y, z)[0],
    "lc_dfs": SolutionDFS().canMeasureWater,
    "lc_enum": SolutionEnum().canMeasureWater,
    "lc_gcd": SolutionGCD().canMeasureWater,
    "lc_ext": SolutionExtGCD().canMeasureWater,
    "leetcode": sol.canMeasureWater,
}


def check_all(x, y, z, expected, label):
    for key, fn in impls.items():
        actual = fn(x, y, z)
        assert actual == expected, (label, key, (x, y, z), actual, expected)
    print("PASS", label)


for idx, (x, y, z, expected) in enumerate(examples, 1):
    check_all(x, y, z, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [24]:
check_all(1, 1, 1, True, "hai bình 1 lít, target 1")
check_all(1, 1, 2, True, "x + y = target (đổ đầy cả hai)")
check_all(1, 1, 3, False, "target > x + y")
check_all(5, 5, 5, True, "x = y = target")
check_all(5, 5, 10, True, "x = y, target = x + y")
check_all(5, 5, 7, False, "x = y, target không là bội của 5")
check_all(2, 4, 6, True, "target = x + y, cùng ước 2")
check_all(2, 4, 8, False, "gcd | target nhưng target > x + y (thử lỗi quên z <= x+y)")
check_all(2, 6, 4, True, "target chẵn, gcd = 2")
check_all(4, 6, 2, True, "target = gcd (2)")
check_all(4, 6, 11, False, "target lẻ và > x + y")
check_all(1, 1000, 1000, True, "x = 1: mọi target <= 1 + y")
check_all(1000, 1000, 1000, True, "cực đại: x = y = target = 1000")
check_all(1000, 999, 1, True, "hai số nguyên tố cùng nhau gần nhau, target = 1")
check_all(997, 991, 1000, True, "hai số nguyên tố, gcd = 1, target <= x + y")
check_all(6, 9, 4, False, "gcd = 3, target = 4 không chia hết 3")
check_all(6, 9, 12, True, "gcd = 3, 12 chia hết 3 và <= 15")

PASS hai bình 1 lít, target 1
PASS x + y = target (đổ đầy cả hai)
PASS target > x + y
PASS x = y = target
PASS x = y, target = x + y
PASS x = y, target không là bội của 5
PASS target = x + y, cùng ước 2
PASS gcd | target nhưng target > x + y (thử lỗi quên z <= x+y)
PASS target chẵn, gcd = 2
PASS target = gcd (2)
PASS target lẻ và > x + y
PASS x = 1: mọi target <= 1 + y
PASS cực đại: x = y = target = 1000
PASS hai số nguyên tố cùng nhau gần nhau, target = 1
PASS hai số nguyên tố, gcd = 1, target <= x + y
PASS gcd = 3, target = 4 không chia hết 3
PASS gcd = 3, 12 chia hết 3 và <= 15


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi hay gặp: (a) **quên `target <= x + y`** (chỉ xét `target % gcd`); (b) **chỉ xét bội của `x` hoặc `y`**.

In [25]:
def wrong_no_cap(x, y, z):
    return z % math.gcd(x, y) == 0                          # SAI: thiếu z <= x + y


def wrong_multiples(x, y, z):
    return z <= x + y and (z % x == 0 or z % y == 0 or z == x + y)    # SAI: bỏ sót tổ hợp như 3a + 5b = 4


print("quên giới hạn:", wrong_no_cap(2, 4, 8), "!=", can_measure_gcd(2, 4, 8))
print("chỉ bội:", wrong_multiples(3, 5, 4), "!=", can_measure_gcd(3, 5, 4))
assert wrong_no_cap(2, 4, 8) != can_measure_gcd(2, 4, 8)
assert wrong_multiples(3, 5, 4) != can_measure_gcd(3, 5, 4)

quên giới hạn: True != False
chỉ bội: False != True


### 12.3. Đối chiếu vét cạn và ngẫu nhiên (có seed)

(1) **Vét cạn** mọi `x, y` trong `1..40` và mọi `target` trong `1..x + y + 3` (BFS là chuẩn). (2) **Ngẫu nhiên** có seed với `x, y` lên đến `1000`, đối chiếu mọi cách với BFS (BFS chỉ thăm khoảng `2(x + y)` trạng thái nên vẫn nhanh).

In [26]:
count = 0
for x in range(1, 41):
    for y in range(1, 41):
        for z in range(1, x + y + 4):
            ref = can_measure_bfs(x, y, z)
            assert can_measure_gcd(x, y, z) == ref, (x, y, z)
            assert can_measure_enum(x, y, z) == ref, (x, y, z)
            assert can_measure_dfs(x, y, z) == ref, (x, y, z)
            assert can_measure_certificate(x, y, z)[0] == ref, (x, y, z)
            assert sol.canMeasureWater(x, y, z) == ref, (x, y, z)
            count += 1
print("PASS vét cạn", count, "bộ (x, y, target) với x, y <= 40")

PASS vét cạn 70400 bộ (x, y, target) với x, y <= 40


In [27]:
rng = random.Random(365)
for trial in range(600):                      # BFS làm chuẩn, toàn miền ràng buộc
    x, y = rng.randint(1, 1000), rng.randint(1, 1000)
    z = rng.randint(1, 1000)
    ref = can_measure_bfs(x, y, z)
    for key, fn in impls.items():
        assert fn(x, y, z) == ref, (key, x, y, z)
print("PASS 600 bộ ngẫu nhiên (x, y, target <= 1000) đối chiếu với BFS")

for trial in range(20000):                    # toàn miền ràng buộc, đối chiếu giữa các cách nhanh
    x, y, z = rng.randint(1, 1000), rng.randint(1, 1000), rng.randint(1, 1000)
    ref = can_measure_gcd(x, y, z)
    assert can_measure_enum(x, y, z) == ref, (x, y, z)
    assert can_measure_certificate(x, y, z)[0] == ref, (x, y, z)
    assert sol.canMeasureWater(x, y, z) == ref
    assert SolutionExtGCD().canMeasureWater(x, y, z) == ref
print("PASS 20000 bộ ngẫu nhiên toàn miền (1..1000) giữa các cách nhanh")

# Một vài cặp lớn đối chiếu với BFS
for x, y, z in [(400, 399, 797), (400, 300, 700), (400, 300, 650), (399, 399, 21), (350, 450, 100), (350, 450, 99)]:
    assert can_measure_bfs(x, y, z) == can_measure_gcd(x, y, z), (x, y, z)
print("PASS các cặp lớn")

PASS 600 bộ ngẫu nhiên (x, y, target <= 1000) đối chiếu với BFS


PASS 20000 bộ ngẫu nhiên toàn miền (1..1000) giữa các cách nhanh
PASS các cặp lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên điều kiện `target <= x + y`

In [28]:
class SaiQuenCap:
    def canMeasureWater(self, x, y, target):
        return target % math.gcd(x, y) == 0        # SAI: thiếu target > x + y


print("sai:", SaiQuenCap().canMeasureWater(2, 4, 8), "| đúng:", Solution().canMeasureWater(2, 4, 8))

sai: True | đúng: False


**Vì sao sai:** `gcd(2, 4) = 2` và `8` chia hết cho `2`, nhưng hai bình cộng lại chỉ chứa `6` lít, không thể có `8`. Bézout chỉ nói về **tổ hợp nguyên** (cho phép đổ đầy nhiều lần), còn bài toán còn **giới hạn sức chứa**. **Cách sửa:** `if target > x + y: return False`.

### Lỗi 2: Chỉ kiểm tra bội của `x` hoặc `y`

In [29]:
class SaiChiBoi:
    def canMeasureWater(self, x, y, target):
        return target <= x + y and (target % x == 0 or target % y == 0 or target == x + y)


print("sai:", SaiChiBoi().canMeasureWater(3, 5, 4), "| đúng:", Solution().canMeasureWater(3, 5, 4))

sai: False | đúng: True


**Vì sao sai:** ví dụ 1 (`3, 5, 4`) cần phối hợp hai bình (`4 = 3·3 − 5`), không phải bội của riêng bình nào. Chỉ xét `target % x` hoặc `target % y` bỏ sót mọi tổ hợp. **Cách sửa:** dùng `gcd(x, y)`: mọi tổ hợp nguyên đều là bội của gcd.

### Lỗi 3: BFS/DFS không có tập `visited`

In [30]:
def sai_bfs_khong_visited(x, y, z, cap=100000):
    q = deque([(0, 0)])
    steps = 0
    while q:
        i, j = q.popleft()
        steps += 1
        if i + j == z:
            return True, steps
        if steps >= cap:                      # chỉ để minh họa: thực tế vòng lặp này KHÔNG BAO GIỜ dừng khi z không đạt được
            return None, steps
        q.extend(next_states(i, j, x, y))     # SAI: thêm cả trạng thái đã thăm
    return False, steps


print(sai_bfs_khong_visited(3, 5, 4))         # (None, 100000): sau 100000 bước vẫn chưa tìm thấy (bản đúng chỉ cần 15 bước, xem mục 8)
print(sai_bfs_khong_visited(2, 6, 5))         # z không đạt được: không bao giờ dừng, bị chặn bởi cap
print("đúng:", can_measure_bfs(2, 6, 5))

(True, 88088)
(None, 100000)
đúng: False


**Vì sao sai:** đồ thị có **chu trình** (đổ đầy rồi đổ cạn quay lại trạng thái cũ), và hàng đợi còn **phình ra theo cấp số nhân** (mỗi trạng thái đẩy 6 trạng thái mới). Khi `target` không đạt được, vòng lặp **không bao giờ kết thúc** (ở đây ta phải đặt `cap` nhân tạo). **Cách sửa:** `seen = set()` và chỉ thêm trạng thái **chưa thăm**.

### Lỗi 4: DFS đệ quy quá sâu → `RecursionError`

In [31]:
def dfs_recursive(x, y, z):
    seen = set()

    def dfs(i, j):
        if (i, j) in seen:
            return False
        seen.add((i, j))
        if i + j == z:
            return True
        a = min(i, y - j)
        b = min(j, x - i)
        return (dfs(x, j) or dfs(i, y) or dfs(0, j) or dfs(i, 0)
                or dfs(i - a, j + a) or dfs(i + b, j - b))

    return dfs(0, 0)


print(dfs_recursive(3, 5, 4), dfs_recursive(2, 6, 5))          # nhỏ: ổn
try:
    dfs_recursive(1000, 998, 1)                                # gcd = 2, target = 1 không đạt được: phải duyệt cả đồ thị
except RecursionError:
    print("RecursionError: đệ quy vượt giới hạn mặc định (khoảng 1000 tầng)")

True False
RecursionError: đệ quy vượt giới hạn mặc định (khoảng 1000 tầng)


**Vì sao sai:** DFS đệ quy đi sâu theo một đường dài tới cỡ số trạng thái đã thăm (hàng nghìn), vượt giới hạn đệ quy của Python (`sys.getrecursionlimit()` thường là `1000`). **Cách sửa:** dùng **ngăn xếp tường minh** (Cách 2), hoặc BFS, hoặc tốt nhất công thức GCD (không duyệt). (Có thể nâng giới hạn bằng `sys.setrecursionlimit`, nhưng dễ làm tràn ngăn xếp C — không khuyến khích.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Hai thùng/bình, đổ — rót — hỏi lượng đo được**" ⇒ nghĩ đến **gcd** và điều kiện `target <= x + y`.
- Bài có **trạng thái nhỏ** (hai số) và **thao tác hữu hạn** ⇒ luôn có thể **BFS/DFS trên đồ thị trạng thái** làm oracle (vét cạn) để kiểm công thức bạn đoán.
- "Tập các số biểu diễn được dạng `a·x + b·y` (với `a, b` nguyên)" = **bội của `gcd(x, y)`** (Bézout). Nếu `a, b` bị giới hạn không âm thì khác (bài đổi tiền, **số Frobenius**).
- Dùng **Euclid** (`while b: a, b = b, a % b`) hoặc `math.gcd` thay vì tự duyệt ước.
- Khi bài có đồ thị có chu trình: **bắt buộc** có `visited`, và đánh dấu **khi thêm** vào hàng đợi/ngăn xếp (không phải khi lấy ra) để mỗi trạng thái vào đúng một lần.
- Với DFS trên đồ thị có thể sâu: ưu tiên **ngăn xếp tường minh** để tránh `RecursionError`.

## 15. Biến thể của bài

1. **Số thao tác ít nhất** để đạt `target` (BFS cho ngay đáp án).
2. **In ra dãy thao tác** (truy vết `parent`, như mục 8).
3. **Đếm số tổng lượng nước có thể đạt** (các bội của `g` không vượt `x + y`: `(x + y) // g + 1` giá trị kể cả `0`).
4. **Ba bình** hoặc nhiều bình: BFS trên tuple dài hơn; công thức gcd mở rộng thành gcd nhiều số nhưng điều kiện cần/đủ phức tạp hơn.
5. **Die Hard** / **Measure water with jugs** trong các cuộc thi: cùng ý tưởng.
6. Bài liên quan: **Bézout / Frobenius coin problem**, **Greatest Common Divisor of Strings (1071)**, **Check If It Is a Good Array (1250)** (mảng "tốt" nếu gcd các phần tử bằng 1).

In [32]:
def min_steps(x, y, z):
    """Số thao tác ít nhất để tổng lượng nước = z, hoặc -1 nếu không thể."""
    dist = {(0, 0): 0}
    q = deque([(0, 0)])
    while q:
        i, j = q.popleft()
        if i + j == z:
            return dist[(i, j)]
        for ns in next_states(i, j, x, y):
            if ns not in dist:
                dist[ns] = dist[(i, j)] + 1
                q.append(ns)
    return -1


def count_reachable_totals(x, y):
    """Số giá trị tổng i + j (kể cả 0) có thể đạt."""
    return (x + y) // math.gcd(x, y) + 1


def brute_reachable_totals(x, y):
    seen = {(0, 0)}
    q = deque([(0, 0)])
    while q:
        i, j = q.popleft()
        for ns in next_states(i, j, x, y):
            if ns not in seen:
                seen.add(ns)
                q.append(ns)
    return len({i + j for i, j in seen})


print(min_steps(3, 5, 4), min_steps(2, 6, 5), min_steps(1, 2, 3))     # số thao tác ít nhất: 7 -1 2
print(count_reachable_totals(3, 5), brute_reachable_totals(3, 5))     # 9 9
for x in range(1, 15):
    for y in range(1, 15):
        assert count_reachable_totals(x, y) == brute_reachable_totals(x, y)
print("công thức đếm tổng đạt được khớp vét cạn")

7 -1 2
9 9
công thức đếm tổng đạt được khớp vét cạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | duyệt đồ thị `(i, j)` theo tầng | `O(x + y)` | `O(x + y)` | oracle, cần số bước/đường đi ngắn nhất |
| 2. DFS (ngăn xếp) | cùng đồ thị, đi sâu trước | `O(x + y)` | `O(x + y)` | chỉ cần có/không, tránh đệ quy sâu |
| 3. Liệt kê `a·x + b·y` | thử `a` trong `0..y−1`, kiểm `(target − a·x) % y` | `O(y)` | `O(1)` | không muốn dùng gcd, vẫn nhanh |
| 4. Công thức GCD | `target <= x + y` và `target % gcd == 0` | `O(log min(x, y))` | `O(1)` | **nên nộp** |
| 5. Euclid mở rộng | dựng hệ số `(a, b)` | `O(log min(x, y))` | `O(log min(x, y))` | cần "bằng chứng"/hệ số cụ thể |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `math.gcd(12, 18)`; (b) `(-5) % 3`; (c) `Solution().canMeasureWater(4, 6, 8)`; (d) `Solution().canMeasureWater(4, 6, 11)`; (e) `Solution().canMeasureWater(4, 6, 3)`.

**Bài 2 (tính tay).** Với `x = 4`, `y = 6`, `target = 2`: tính `gcd`, rồi viết **chuỗi thao tác và trạng thái `(i, j)`** để tổng lượng nước bằng `2` (dùng các tên thao tác `fill1/fill2/empty1/empty2/pour12/pour21` ở mục 4).

**Bài 3 (điền TODO).** Hoàn thành `can_measure_todo(x, y, target)` theo công thức GCD (dùng vòng lặp Euclid tự viết).

In [33]:
def can_measure_todo(x, y, target):
    # TODO 1: nếu target > x + y thì trả về False
    # TODO 2: tính g = gcd(x, y) bằng vòng lặp Euclid: while b: a, b = b, a % b
    # TODO 3: trả về target % g == 0
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `can_measure_bfs_path(x, y, target)` trả về **danh sách các trạng thái** từ `(0, 0)` đến trạng thái có tổng `target` (đường đi ngắn nhất), hoặc `None` nếu không thể. Gợi ý: dùng `dict parent` như mục 8.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_steps_ans(x, y, target)` (số thao tác ít nhất) và kiểm tra `min_steps_ans(3, 5, 4) == 7`. (b) Viết `reachable_totals(x, y)` trả về **tập** các tổng đạt được và chứng minh bằng code rằng nó đúng bằng `{0, g, 2g, ..., ≤ x + y}` với `x, y <= 12`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6`; `1` (vì `%` của Python luôn không âm); `True` (`gcd = 2`, `8 <= 10`, `8` chẵn); `False` (`11 > 10`); `False` (`3` lẻ mà `gcd = 2`). Bài 3: bước 1 viết `if target > x + y: return False`.
- **Mức 2 (vừa):** Bài 2: `gcd(4, 6) = 2`. Thử: đổ đầy bình 6 `(0, 6)`; rót sang bình 4 `(4, 2)`; đổ cạn bình 4 `(0, 2)` ⇒ tổng `2`. Bài 4: `parent[start] = None`; khi gặp `i + j == target`, đi ngược `parent` rồi đảo danh sách.
- **Mức 3 (sâu):** Bài 5(a): BFS với `dist`, trả `dist[(i, j)]` ngay khi `i + j == target`. 5(b): BFS toàn bộ, gom `{i + j}`; so với `set(range(0, x + y + 1, g))` (các bội của `g` không vượt `x + y`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert math.gcd(12, 18) == 6
assert (-5) % 3 == 1
assert Solution().canMeasureWater(4, 6, 8) is True
assert Solution().canMeasureWater(4, 6, 11) is False
assert Solution().canMeasureWater(4, 6, 3) is False

# Bài 2: x = 4, y = 6, target = 2
state = (0, 0)
for op in ["fill2", "pour21", "empty1"]:
    state = apply_op(state, op, 4, 6)
    print(op, "->", state)
assert state == (0, 2) and sum(state) == 2 and math.gcd(4, 6) == 2


# Bài 3
def can_measure_todo_ans(x, y, target):
    if target > x + y:
        return False
    a, b = x, y
    while b:
        a, b = b, a % b
    return target % a == 0


for x, y, z, expected in examples:
    assert can_measure_todo_ans(x, y, z) == expected
for x in range(1, 25):
    for y in range(1, 25):
        for z in range(1, x + y + 3):
            assert can_measure_todo_ans(x, y, z) == can_measure_bfs(x, y, z)


# Bài 4
def can_measure_bfs_path(x, y, target):
    start = (0, 0)
    parent = {start: None}
    q = deque([start])
    while q:
        cur = q.popleft()
        if sum(cur) == target:
            path = []
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        for ns in next_states(cur[0], cur[1], x, y):
            if ns not in parent:
                parent[ns] = cur
                q.append(ns)
    return None


p = can_measure_bfs_path(3, 5, 4)
assert p[0] == (0, 0) and sum(p[-1]) == 4
for u, v in zip(p, p[1:]):
    assert v in next_states(u[0], u[1], 3, 5)         # mỗi bước là một thao tác hợp lệ
assert can_measure_bfs_path(2, 6, 5) is None


# Bài 5
def min_steps_ans(x, y, target):
    dist = {(0, 0): 0}
    q = deque([(0, 0)])
    while q:
        cur = q.popleft()
        if sum(cur) == target:
            return dist[cur]
        for ns in next_states(cur[0], cur[1], x, y):
            if ns not in dist:
                dist[ns] = dist[cur] + 1
                q.append(ns)
    return -1


assert min_steps_ans(3, 5, 4) == 7 == min_steps(3, 5, 4)
assert min_steps_ans(2, 6, 5) == -1


def reachable_totals(x, y):
    seen = {(0, 0)}
    q = deque([(0, 0)])
    while q:
        cur = q.popleft()
        for ns in next_states(cur[0], cur[1], x, y):
            if ns not in seen:
                seen.add(ns)
                q.append(ns)
    return {i + j for i, j in seen}


for x in range(1, 13):
    for y in range(1, 13):
        g = math.gcd(x, y)
        assert reachable_totals(x, y) == set(range(0, x + y + 1, g)), (x, y)
print("Tất cả đáp án đều PASS")

fill2 -> (0, 6)
pour21 -> (4, 2)
empty1 -> (0, 2)


Tất cả đáp án đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (GCD)

```python
import math


class Solution:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        if target > x + y:
            return False
        return target % math.gcd(x, y) == 0
```

**Template BFS** (khi cần đường đi / số bước): trạng thái `(i, j)`, 6 thao tác, `seen = set()`, `deque`.

**Checklist**

- [ ] Có điều kiện `target <= x + y` (sức chứa) **và** `target % gcd(x, y) == 0` (Bézout)?
- [ ] Đã kiểm tra các biên: `x = y`, `target = x + y`, `target > x + y` nhưng chia hết gcd, `x = 1`?
- [ ] Nếu duyệt đồ thị: có `visited`, đánh dấu khi thêm vào hàng đợi/ngăn xếp, tránh đệ quy quá sâu?
- [ ] Phép rót dùng `min(nước_có, chỗ_trống)` cho cả hai chiều?
- [ ] Đã đối chiếu công thức với BFS vét cạn trên mọi `x, y` nhỏ?
- [ ] Skeleton đúng `class Solution` / `def canMeasureWater(self, x: int, y: int, target: int) -> bool`?